In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, regexp_replace
from pyspark.sql.functions import sum as spark_sum

### Objetivo do Projeto
Com a proximidade das eleições de 2026, a disponibilidade de informações sobre candidaturas, despesas eleitorais e eleitorado - temos informações suficientes analisar de forma mais estruturada e clara os custos envolvidos no processo eleitoral.

Dessa forma, o trabalho busca consolidar e tornar mais objetivas informações públicas, permitindo observar os custos associados às candidaturas e sua distribuição por estados, categorias de despesas e eleitorado, contribuindo para uma análise mais clara dos dados disponibilizados à população.

### Perguntas de Negócio

Com base nesse contexto, o objetivo deste trabalho é responder os seguintes questionamentos:

**1.** Quais são os 5 estados com maior valor total de despesas eleitorais contratadas em 2026?

**1.1.** Quais são os 3 candidatos com maior valor total de despesas contratadas em cada um desses 5 estados?

**2.** Quais são as 3 principais origens das despesas eleitorais contratadas em 2026?

**2.1.** Quais são as 3 principais descrições de despesas dentro de cada uma das 3 principais origens?

**3.** Qual é o custo das despesas eleitorais contratadas, por eleitor, em cada estado brasileiro em 2026?

### Licença e uso dos dados

As bases utilizadas neste projeto são disponibilizadas publicamente pelo Tribunal Superior Eleitoral (TSE) por meio de seu Portal de Dados Abertos e estão identificadas no portal sob a licença **Creative Commons Atribuição (CC BY)**. 

### Coleta e carregamento de dados
Foram utilizadas três bases disponibilizadas pelo portal de dados abertos do TSE:

- Base de candidaturas - https://dadosabertos.tse.jus.br/dataset/candidatos-2026#
- Base de despesas eleitorais contratadas (prestação de contas até o dia 24/09/2026) - https://dadosabertos.tse.jus.br/dataset/prestacao-de-contas-eleitorais-2026#
- Base de perfil do eleitorado - https://dadosabertos.tse.jus.br/dataset/eleitorado-atual#

Os três arquivos foram armazenados manualmente no volume do Databricks abaixo, tendo em vista o tamanho das tabelas e o objetivo principal para responder os questionamentos acima. 

`/Volumes/workspace/default/MVP_Engenharia_de_Dados`

Devido ao tamanho dos arquivos (+2GB) não será possível disponibilizá-los pelo GITHUB, que limita os uploads em até 25mb.

Todos os três resultados representam o snapshot dos arquivos disponíveis na data de execução do projeto

### Carga e pipeline
O pipeline segue o fluxo:

`Fonte TSE → Bronze → Silver → Gold → SQL / Análise`

A camada Bronze preserva os dados recebidos da fonte. A camada Silver realiza padronização, conversão de tipos e preparação dos dados. A camada Gold consolida estruturas analíticas utilizadas nas perguntas de negócio.

### Ingestão e tratamento da Tabela de Candidatos
Leitura do arquivo original e posterior transformação para a camada Silver.

In [0]:
from pyspark.sql.functions import col, trim, upper

caminho = "/Volumes/workspace/default/MVP_Engenharia_de_Dados"

df_candidatos = (
    spark.read
    .option("header", True)
    .option("delimiter", ";")
    .option("quote", '"')
    .option("encoding", "UTF-8")
    .csv(f"{caminho}/consulta_cand_2026_BRASIL.csv")
)

display(df_candidatos.limit(5))

### Tratamento da Tabela de Candidatos (Silver)

Após a ingestão dos dados na camada Bronze, foram realizados os seguintes processos de padronização:

- Conversão do identificador do candidato para o tipo numérico inteiro.
- Remoção de espaços excedentes nos campos textuais.
- Padronização dos campos de partido e estado para letras maiúsculas.
- Renomeação dos campos para uma nomenclatura padronizada na camada Silver.
- Estruturação dos dados com uma linha por candidato, utilizando `ID_CANDIDATO` como identificador da candidatura.

In [0]:
df_candidatos_silver = df_candidatos.select(
    col("SQ_CANDIDATO").cast("long").alias("ID_CANDIDATO"),
    trim(col("NM_CANDIDATO")).alias("CANDIDATO"),
    upper(trim(col("SG_PARTIDO"))).alias("PARTIDO"),
    trim(col("DS_CARGO")).alias("CARGO"),
    upper(trim(col("SG_UF"))).alias("ESTADO")
)

display(df_candidatos_silver.limit(10))

### Ingestão e tratamento da Tabela de Despesas
Leitura do arquivo original e posterior transformação para a camada Silver, incluindo tratamento dos valores monetários e datas.

In [0]:
df_despesas = (
    spark.read
    .option("header", True)
    .option("delimiter", ";")
    .option("quote", '"')
    .option("encoding", "UTF-8")
    .csv(f"{caminho}/despesas_contratadas_candidatos_2026_BRASIL.csv")
)

display(df_despesas.limit(5))

### Tratamento da Tabela de Despesas (Silver)

Após a ingestão dos dados na camada Bronze, foram realizados os seguintes processos de padronização:

- Conversão dos identificadores de despesa e candidato para o tipo numérico inteiro.
- Padronização do estado e remoção de espaços excedentes nos campos textuais.
- Conversão dos valores de despesas para o formato numérico decimal.
- Conversão da data da despesa para o formato de data.
- Renomeação e padronização dos campos para a estrutura da `FACT_DESPESA`.
- Manutenção dos registros da fonte, sem exclusão de dados durante a transformação para a camada Silver.

In [0]:
from pyspark.sql.functions import col, trim, upper, to_date, regexp_replace

df_despesas_silver = df_despesas.select(

    col("SQ_DESPESA")
        .cast("long")
        .alias("ID_DESPESA"),

    col("SQ_CANDIDATO")
        .cast("long")
        .alias("ID_CANDIDATO"),

    upper(trim(col("SG_UF")))
        .alias("ESTADO"),

    trim(col("DS_ORIGEM_DESPESA"))
        .alias("CATEGORIA_DESPESA"),

    trim(col("DS_DESPESA"))
        .alias("DESPESA_DETALHADA"),

    regexp_replace(
        regexp_replace(
            trim(col("VR_DESPESA_CONTRATADA")),
            "\\.",
            ""
        ),
        ",",
        "."
    )
    .cast("decimal(18,2)")
    .alias("VALOR_DESPESA"),

    to_date(
        trim(col("DT_DESPESA")),
        "dd/MM/yyyy"
    )
    .alias("DATA_DESPESA")

)

display(df_despesas_silver.limit(10))

### Ingestão e tratamento da Tabela de Eleitorado
Leitura do arquivo original e posterior transformação para a camada Silver.

In [0]:
caminho = "/Volumes/workspace/default/MVP_Engenharia_de_Dados"

df_eleitorado = (
    spark.read
    .option("header", True)
    .option("delimiter", ";")
    .option("quote", '"')
    .option("encoding", "ISO-8859-1")
    .csv(f"{caminho}/perfil_eleitorado_ATUAL.csv")
)

display(df_eleitorado.limit(5))

### Tratamento da Tabela de Eleitorado (Silver)

Após a ingestão dos dados na camada Bronze, foram realizados os seguintes processos de padronização:

- Conversão dos códigos de município e zona eleitoral para tipos numéricos.
- Padronização do estado para letras maiúsculas e remoção de espaços excedentes.
- Conversão da data de geração para o formato de data.
- Remoção de espaços excedentes nos campos textuais relacionados ao perfil do eleitorado.
- Conversão das quantidades de eleitores para tipos numéricos inteiros.
- Renomeação dos campos para uma nomenclatura padronizada na camada Silver.

In [0]:
df_eleitorado_silver = df_eleitorado.select(

    to_date(col("DT_GERACAO"), "dd/MM/yyyy").alias("DATA_GERACAO"),

    upper(trim(col("SG_UF"))).alias("ESTADO"),

    col("CD_MUNICIPIO").cast("int").alias("CODIGO_MUNICIPIO"),

    trim(col("NM_MUNICIPIO")).alias("MUNICIPIO"),

    col("NR_ZONA").cast("int").alias("ZONA"),

    trim(col("DS_GENERO")).alias("GENERO"),

    trim(col("DS_ESTADO_CIVIL")).alias("ESTADO_CIVIL"),

    trim(col("DS_FAIXA_ETARIA")).alias("FAIXA_ETARIA"),

    trim(col("DS_GRAU_INSTRUCAO")).alias("GRAU_INSTRUCAO"),

    trim(col("DS_COR_RACA")).alias("COR_RACA"),

    trim(col("TP_OBRIGATORIEDADE_VOTO")).alias("OBRIGATORIEDADE_VOTO"),

    col("QT_ELEITORES").cast("long").alias("ELEITORES"),

    col("QT_ELEITORES_BIOMETRIA").cast("long").alias("ELEITORES_BIOMETRIA"),

    col("QT_ELEITORES_DEFICIENCIA").cast("long").alias("ELEITORES_DEFICIENCIA"),

    col("QT_ELEITORES_NOME_SOCIAL").cast("long").alias("ELEITORES_NOME_SOCIAL")

)

display(df_eleitorado_silver.limit(10))

### Modelagem de dados
O projeto foi estruturado utilizando um **esquema estrela**, com a `FACT_DESPESA` como tabela fato central e dimensões relacionadas às principais análises do projeto.

**FACT_DESPESA**  
Tabela fato central, com uma linha por registro de despesa disponibilizado pela fonte. Contém as medidas de valor e as chaves utilizadas para relacionamento com as dimensões.

**DIM_CANDIDATO**  
Uma linha por candidato, com `ID_CANDIDATO` como chave primária. Armazena os atributos descritivos utilizados nas análises de candidatos.

**DIM_ESTADO**  
Uma linha por estado. Além da identificação da unidade federativa, contém o total de eleitores obtido pela agregação da base de eleitorado por estado.

**DIM_CATEGORIA_DESPESA**  
Uma linha por combinação de categoria e descrição de despesa, concentrando os atributos descritivos utilizados na análise das origens e tipos de gastos.

**DIM_DATA**  
Uma linha por data de despesa, utilizada para organizar e filtrar as informações temporais da `FACT_DESPESA`.

**Relacionamentos:** `FACT_DESPESA` se relaciona com `DIM_CANDIDATO`, `DIM_ESTADO`, `DIM_CATEGORIA_DESPESA` e `DIM_DATA`, formando o esquema estrela utilizado na camada analítica.

**Tratamento do eleitorado:** a tabela de eleitorado permanece como fonte Silver para a modelagem. Seus registros são agregados por `ESTADO` para formar a `DIM_ESTADO`, evitando o relacionamento direto entre o nível detalhado do eleitorado e o nível de detalhe das despesas.

**Diagrama do modelo:** Abaixo está a representação do esquema estrela utilizado no projeto.

![Diagrama do modelo](<./Imagens de Catálogos/Diagrama Estrela.png>)

### Catálogo de dados

O catálogo de dados abaixo apresenta os principais atributos das tabelas utilizadas no modelo dimensional, relacionando os campos originais às estruturas modeladas durante a camada Silver - com seus tipos, regras de transformação e respectivas funções no modelo.

### DIM_CANDIDATO

![Catálogo DIM_CANDIDATO](<./Imagens de Catálogos/DIM_CANDIDATO.png>)

### FACT_DESPESA

![Catálogo FACT_DESPESA](<./Imagens de Catálogos/FACT_DESPESA.png>)

### DIM_ESTADO

![Catálogo DIM_ESTADO](<./Imagens de Catálogos/DIM_ESTADO.png>)

### DIM_CATEGORIA_DESPESA

![Catálogo DIM_CATEGORIA_DESPESA](<./Imagens de Catálogos/DIM_CATEGORIA_DESPESA.png>)

### DIM_DATA

![Catálogo DIM_DATA](<./Imagens de Catálogos/DIM DATA.png>)

### Validação das camadas
Conferência da quantidade de registros entre Bronze e Silver.

In [0]:
print("Candidatos Bronze:", df_candidatos.count())
print("Candidatos Silver:", df_candidatos_silver.count())

print("Despesas Bronze:", df_despesas.count())
print("Despesas Silver:", df_despesas_silver.count())

print("Eleitorado Bronze:", df_eleitorado.count())
print("Eleitorado Silver:", df_eleitorado_silver.count())

### Validação do valor total de despesas
Conferência do valor monetário após a transformação para decimal.

In [0]:
from pyspark.sql.functions import sum as spark_sum

df_despesas_silver.select(
    spark_sum("VALOR_DESPESA").alias("TOTAL_DESPESAS")
).show()

### Views de apoio
Criação das views temporárias utilizadas nas análises SQL.

In [0]:
df_despesas_silver.createOrReplaceTempView("despesas_silver")
df_candidatos_silver.createOrReplaceTempView("candidatos_silver")
df_eleitorado_silver.createOrReplaceTempView("eleitorado_silver")
df_eleitorado.createOrReplaceTempView("eleitorado_bronze")

### Qualidade de dados
Aqui foram realizados testes de completude, unicidade, consistência, integridade referencial e identificação de anomalias nas bases utilizadas.

### Qualidade da tabela de despesas (Silver)

#### Volume e unicidade

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS,
    COUNT(DISTINCT ID_DESPESA) AS IDS_DISTINTOS,
    COUNT(*) - COUNT(DISTINCT ID_DESPESA) AS POSSIVEIS_DUPLICIDADES
FROM despesas_silver;

#### Completude

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS,

    SUM(CASE WHEN ID_DESPESA IS NULL THEN 1 ELSE 0 END) AS NULOS_ID_DESPESA,
    SUM(CASE WHEN ID_CANDIDATO IS NULL THEN 1 ELSE 0 END) AS NULOS_ID_CANDIDATO,
    SUM(CASE WHEN ESTADO IS NULL THEN 1 ELSE 0 END) AS NULOS_ESTADO,
    SUM(CASE WHEN CATEGORIA_DESPESA IS NULL THEN 1 ELSE 0 END) AS NULOS_CATEGORIA,
    SUM(CASE WHEN DESPESA_DETALHADA IS NULL THEN 1 ELSE 0 END) AS NULOS_DESPESA,
    SUM(CASE WHEN VALOR_DESPESA IS NULL THEN 1 ELSE 0 END) AS NULOS_VALOR,
    SUM(CASE WHEN DATA_DESPESA IS NULL THEN 1 ELSE 0 END) AS NULOS_DATA

FROM despesas_silver;

#### Registros anômalos

In [0]:
%sql
SELECT
    ID_DESPESA,
    DATA_DESPESA,
    VALOR_DESPESA,
    DESPESA_DETALHADA
FROM despesas_silver
WHERE ID_DESPESA = -1;

#### Quantidade de registros anômalos

In [0]:
%sql
SELECT
    COUNT(*) AS QTD_REGISTROS_ANOMALOS
FROM despesas_silver
WHERE ID_DESPESA = -1;

#### Integridade referencial

In [0]:
%sql
SELECT
    COUNT(*) AS DESPESAS_SEM_CANDIDATO
FROM despesas_silver d
LEFT JOIN candidatos_silver c
    ON d.ID_CANDIDATO = c.ID_CANDIDATO
WHERE c.ID_CANDIDATO IS NULL;

#### IDs repetidos

In [0]:
%sql
SELECT
    ID_DESPESA,
    COUNT(*) AS QTD_REGISTROS
FROM despesas_silver
WHERE ID_DESPESA <> -1
GROUP BY ID_DESPESA
HAVING COUNT(*) > 1
ORDER BY QTD_REGISTROS DESC;

#### Quantidade de IDs repetidos

In [0]:
%sql
SELECT
    COUNT(*) AS IDS_DUPLICADOS
FROM (
    SELECT
        ID_DESPESA
    FROM despesas_silver
    WHERE ID_DESPESA <> -1
    GROUP BY ID_DESPESA
    HAVING COUNT(*) > 1
);

#### Investigação de IDs repetidos

In [0]:
%sql
SELECT
    ID_DESPESA,
    COUNT(*) AS QTD_REGISTROS
FROM despesas_silver
WHERE ID_DESPESA <> -1
GROUP BY ID_DESPESA
HAVING COUNT(*) > 1
ORDER BY QTD_REGISTROS DESC
LIMIT 20;

#### Exemplo de ocorrência

In [0]:
%sql
SELECT
    *
FROM despesas_silver
WHERE ID_DESPESA = 73302255;

#### Registro sem candidato

In [0]:
%sql
SELECT
    d.ID_DESPESA,
    d.ID_CANDIDATO,
    d.ESTADO,
    d.CATEGORIA_DESPESA,
    d.DESPESA_DETALHADA,
    d.VALOR_DESPESA,
    d.DATA_DESPESA
FROM despesas_silver d
LEFT JOIN candidatos_silver c
    ON d.ID_CANDIDATO = c.ID_CANDIDATO
WHERE c.ID_CANDIDATO IS NULL;

### Conclusão — Despesas Eleitorais

A tabela de despesas na camada Silver possui **767.497 registros**. Foram identificados **4.967 registros anômalos** com `ID_DESPESA = -1`, data nula, valor `0,00` e descrição `#NULO`. Esses registros foram preservados na Silver, mantendo a rastreabilidade dos dados da fonte.

Também foram identificados **37.772 IDs de despesa com múltiplas ocorrências**, desconsiderando `ID_DESPESA = -1`. A análise das repetições demonstra que um mesmo identificador pode estar associado a diferentes registros, descrições e valores. Por esse motivo, `ID_DESPESA` não foi considerado uma chave primária simples na modelagem.

Na análise de integridade referencial, foram identificados **168 registros de despesas sem correspondência na tabela de candidatos**. Esses registros foram mantidos na Silver e documentados como uma limitação de integridade das fontes de dados.

### Qualidade da tabela de candidatos

#### Volume e unicidade

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS,
    COUNT(DISTINCT ID_CANDIDATO) AS IDS_DISTINTOS,
    COUNT(*) - COUNT(DISTINCT ID_CANDIDATO) AS POSSIVEIS_DUPLICIDADES
FROM candidatos_silver;

#### Completude

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS,

    SUM(CASE WHEN ID_CANDIDATO IS NULL THEN 1 ELSE 0 END) AS NULOS_ID_CANDIDATO,
    SUM(CASE WHEN CANDIDATO IS NULL THEN 1 ELSE 0 END) AS NULOS_CANDIDATO,
    SUM(CASE WHEN PARTIDO IS NULL THEN 1 ELSE 0 END) AS NULOS_PARTIDO,
    SUM(CASE WHEN CARGO IS NULL THEN 1 ELSE 0 END) AS NULOS_CARGO,
    SUM(CASE WHEN ESTADO IS NULL THEN 1 ELSE 0 END) AS NULOS_ESTADO

FROM candidatos_silver;

#### Duplicidade de ID_CANDIDATO

In [0]:
%sql
SELECT
    ID_CANDIDATO,
    COUNT(*) AS QTD
FROM candidatos_silver
GROUP BY ID_CANDIDATO
HAVING COUNT(*) > 1
ORDER BY QTD DESC;

#### Conclusão — candidatos
A tabela de candidatos na camada Silver possui **20.836 registros**, com **20.836 `ID_CANDIDATO` distintos** e sem nulos nos campos analisados. Não foram identificadas duplicidades da chave primária.

### Qualidade da tabela de eleitorado (Silver)

#### Volume

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS
FROM eleitorado_silver;

#### Completude

In [0]:
%sql
SELECT
    COUNT(*) AS TOTAL_REGISTROS,
    SUM(CASE WHEN DATA_GERACAO IS NULL THEN 1 ELSE 0 END) AS NULOS_DATA_GERACAO,
    SUM(CASE WHEN ESTADO IS NULL THEN 1 ELSE 0 END) AS NULOS_ESTADO,
    SUM(CASE WHEN CODIGO_MUNICIPIO IS NULL THEN 1 ELSE 0 END) AS NULOS_CODIGO_MUNICIPIO,
    SUM(CASE WHEN MUNICIPIO IS NULL THEN 1 ELSE 0 END) AS NULOS_MUNICIPIO,
    SUM(CASE WHEN ZONA IS NULL THEN 1 ELSE 0 END) AS NULOS_ZONA,
    SUM(CASE WHEN GENERO IS NULL THEN 1 ELSE 0 END) AS NULOS_GENERO,
    SUM(CASE WHEN ESTADO_CIVIL IS NULL THEN 1 ELSE 0 END) AS NULOS_ESTADO_CIVIL,
    SUM(CASE WHEN FAIXA_ETARIA IS NULL THEN 1 ELSE 0 END) AS NULOS_FAIXA_ETARIA,
    SUM(CASE WHEN GRAU_INSTRUCAO IS NULL THEN 1 ELSE 0 END) AS NULOS_GRAU_INSTRUCAO,
    SUM(CASE WHEN COR_RACA IS NULL THEN 1 ELSE 0 END) AS NULOS_COR_RACA,
    SUM(CASE WHEN OBRIGATORIEDADE_VOTO IS NULL THEN 1 ELSE 0 END) AS NULOS_OBRIGATORIEDADE_VOTO,
    SUM(CASE WHEN ELEITORES IS NULL THEN 1 ELSE 0 END) AS NULOS_ELEITORES,
    SUM(CASE WHEN ELEITORES_BIOMETRIA IS NULL THEN 1 ELSE 0 END) AS NULOS_ELEITORES_BIOMETRIA,
    SUM(CASE WHEN ELEITORES_DEFICIENCIA IS NULL THEN 1 ELSE 0 END) AS NULOS_ELEITORES_DEFICIENCIA,
    SUM(CASE WHEN ELEITORES_NOME_SOCIAL IS NULL THEN 1 ELSE 0 END) AS NULOS_ELEITORES_NOME_SOCIAL
FROM eleitorado_silver;

#### Granularidade — teste exploratório

In [0]:
%sql
SELECT
    DATA_GERACAO,
    ESTADO,
    CODIGO_MUNICIPIO,
    ZONA,
    GENERO,
    ESTADO_CIVIL,
    FAIXA_ETARIA,
    GRAU_INSTRUCAO,
    COR_RACA,
    OBRIGATORIEDADE_VOTO,
    COUNT(*) AS QTD
FROM eleitorado_silver
GROUP BY
    DATA_GERACAO,
    ESTADO,
    CODIGO_MUNICIPIO,
    ZONA,
    GENERO,
    ESTADO_CIVIL,
    FAIXA_ETARIA,
    GRAU_INSTRUCAO,
    COR_RACA,
    OBRIGATORIEDADE_VOTO
HAVING COUNT(*) > 1
ORDER BY QTD DESC;

#### Consulta à Bronze para investigação

In [0]:
%sql
SELECT *
FROM eleitorado_bronze
WHERE SG_UF = 'AC'
  AND CD_MUNICIPIO = 1007
  AND NR_ZONA = 9
LIMIT 20;

#### Valores mínimos das medidas

In [0]:
%sql
SELECT
    MIN(ELEITORES) AS MIN_ELEITORES,
    MIN(ELEITORES_BIOMETRIA) AS MIN_ELEITORES_BIOMETRIA,
    MIN(ELEITORES_DEFICIENCIA) AS MIN_ELEITORES_DEFICIENCIA,
    MIN(ELEITORES_NOME_SOCIAL) AS MIN_ELEITORES_NOME_SOCIAL
FROM eleitorado_silver;

#### Valores negativos

In [0]:
%sql
SELECT
    COUNT(*) AS REGISTROS_COM_VALOR_NEGATIVO
FROM eleitorado_silver
WHERE ELEITORES < 0
   OR ELEITORES_BIOMETRIA < 0
   OR ELEITORES_DEFICIENCIA < 0
   OR ELEITORES_NOME_SOCIAL < 0;

#### Conclusão — eleitorado
A tabela de eleitorado na camada Silver possui **1.105.719 registros**, sem nulos nos campos analisados e sem valores negativos nas medidas.

A base apresenta **granularidade multidimensional**, combinando atributos geográficos e demográficos, não sendo identificada uma chave primária simples. Os registros foram preservados na Silver e utilizados para compor a `DIM_ESTADO`, por meio da agregação por `ESTADO`.

### Construção da camada Gold — esquema estrela
A partir das tabelas Silver, são construídas as dimensões e a tabela fato que formam o esquema estrela da camada Gold.

**DIM_CANDIDATO:** concentra os atributos descritivos das candidaturas.

**DIM_ESTADO:** contém uma linha por estado e o total de eleitores obtido pela agregação da base de eleitorado.

**DIM_CATEGORIA_DESPESA:** concentra as categorias e descrições das despesas.

**DIM_DATA:** contém uma linha por data de despesa, com atributos temporais derivados.

**FACT_DESPESA:** contém os registros de despesas e a medida `VALOR_DESPESA`, relacionando-se às quatro dimensões.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, row_number, lit, coalesce, year, month, dayofmonth, quarter
from pyspark.sql.window import Window

#### DIM_CANDIDATO

In [0]:
df_dim_candidato = (
    df_candidatos_silver
    .select("ID_CANDIDATO", "CANDIDATO", "PARTIDO", "CARGO", "ESTADO")
)

#### DIM_ESTADO
Agregação do eleitorado por estado para incorporar o total de eleitores à dimensão.

In [0]:
df_eleitorado_estado = (
    df_eleitorado_silver
    .groupBy("ESTADO")
    .agg(spark_sum("ELEITORES").alias("TOTAL_ELEITORES"))
)

estados_fact = df_despesas_silver.select("ESTADO").distinct()

w_estado = Window.orderBy("ESTADO")

df_dim_estado = (
    estados_fact
    .join(df_eleitorado_estado, on="ESTADO", how="left")
    .withColumn("TOTAL_ELEITORES", coalesce(col("TOTAL_ELEITORES"), lit(0).cast("long")))
    .withColumn("ID_ESTADO", row_number().over(w_estado))
    .select("ID_ESTADO", "ESTADO", "TOTAL_ELEITORES")
)

#### DIM_CATEGORIA_DESPESA
Uma linha por combinação de categoria e descrição de despesa.

In [0]:
w_categoria = Window.orderBy("CATEGORIA_DESPESA", "DESPESA_DETALHADA")

df_dim_categoria_despesa = (
    df_despesas_silver
    .select("CATEGORIA_DESPESA", "DESPESA_DETALHADA")
    .dropDuplicates()
    .withColumn("ID_CATEGORIA_DESPESA", row_number().over(w_categoria))
    .select(
        "ID_CATEGORIA_DESPESA",
        "CATEGORIA_DESPESA",
        "DESPESA_DETALHADA"
    )
)

#### DIM_DATA
Uma linha por data de despesa, com atributos temporais derivados.

In [0]:
from pyspark.sql.functions import col, date_format, year, month, dayofmonth, quarter
from pyspark.sql.types import StructType, StructField, LongType, DateType, IntegerType

df_dim_data = (
    df_despesas_silver
    .select("DATA_DESPESA")
    .where(col("DATA_DESPESA").isNotNull())
    .dropDuplicates()
    .withColumn(
        "ID_DATA",
        date_format("DATA_DESPESA", "yyyyMMdd").cast("long")
    )
    .withColumn("ANO", year("DATA_DESPESA"))
    .withColumn("MES", month("DATA_DESPESA"))
    .withColumn("DIA", dayofmonth("DATA_DESPESA"))
    .withColumn("TRIMESTRE", quarter("DATA_DESPESA"))
    .select(
        "ID_DATA",
        "DATA_DESPESA",
        "ANO",
        "MES",
        "DIA",
        "TRIMESTRE"
    )
)

schema_dim_data = StructType([
    StructField("ID_DATA", LongType(), False),
    StructField("DATA_DESPESA", DateType(), True),
    StructField("ANO", IntegerType(), True),
    StructField("MES", IntegerType(), True),
    StructField("DIA", IntegerType(), True),
    StructField("TRIMESTRE", IntegerType(), True)
])

df_data_desconhecida = spark.createDataFrame(
    [(0, None, None, None, None, None)],
    schema_dim_data
)

df_dim_data = df_data_desconhecida.unionByName(df_dim_data)

display(df_dim_data.limit(10))

#### FACT_DESPESA
A tabela fato recebe as chaves das dimensões por meio dos relacionamentos com as tabelas dimensionais.

In [0]:
# Dimensão candidata com registro desconhecido para preservar o registro órfão identificado na Silver.
df_candidato_desconhecido = spark.createDataFrame(
    [(-1, "CANDIDATO NÃO IDENTIFICADO", None, None, None)],
    schema=df_dim_candidato.schema
)

df_dim_candidato = (
    df_dim_candidato
    .unionByName(df_candidato_desconhecido)
)

df_fact_despesa = (
    df_despesas_silver.alias("f")
    .join(
        df_dim_estado.alias("e"),
        col("f.ESTADO") == col("e.ESTADO"),
        "left"
    )
    .join(
        df_dim_categoria_despesa.alias("d"),
        (col("f.CATEGORIA_DESPESA") == col("d.CATEGORIA_DESPESA")) &
        (col("f.DESPESA_DETALHADA") == col("d.DESPESA_DETALHADA")),
        "left"
    )
    .join(
        df_dim_data.alias("dt"),
        col("f.DATA_DESPESA") == col("dt.DATA_DESPESA"),
        "left"
    )
    .join(
        df_dim_candidato.alias("c"),
        col("f.ID_CANDIDATO") == col("c.ID_CANDIDATO"),
        "left"
    )
    .select(
        col("f.ID_DESPESA"),
        coalesce(col("c.ID_CANDIDATO"), lit(-1)).alias("ID_CANDIDATO"),
        col("e.ID_ESTADO"),
        col("d.ID_CATEGORIA_DESPESA"),
        coalesce(col("dt.ID_DATA"), lit(0)).alias("ID_DATA"),
        col("f.VALOR_DESPESA")
    )
)

display(df_fact_despesa.limit(10))

#### Validação das Estruturas Gold

In [0]:
display(df_dim_candidato.limit(10))
display(df_dim_estado.orderBy("ESTADO"))
display(df_dim_categoria_despesa.limit(10))
display(df_dim_data.limit(10))
display(df_fact_despesa.limit(10))

In [0]:
df_dim_candidato.createOrReplaceTempView("dim_candidato")
df_dim_estado.createOrReplaceTempView("dim_estado")
df_dim_categoria_despesa.createOrReplaceTempView("dim_categoria_despesa")
df_dim_data.createOrReplaceTempView("dim_data")
df_fact_despesa.createOrReplaceTempView("fact_despesa")

### Preparação da Gold Estadual
Para as análises estaduais, a `FACT_DESPESA` é agregada por `DIM_ESTADO`, utilizando o total de eleitores da dimensão.

In [0]:
df_gold_estado = (
    df_fact_despesa.alias("f")
    .join(
        df_dim_estado.alias("e"),
        col("f.ID_ESTADO") == col("e.ID_ESTADO"),
        "left"
    )
    .filter(col("e.ESTADO") != "BR")
    .groupBy(
        col("e.ID_ESTADO"),
        col("e.ESTADO"),
        col("e.TOTAL_ELEITORES")
    )
    .agg(
        spark_sum("f.VALOR_DESPESA").alias("TOTAL_DESPESAS")
    )
)

In [0]:
display(
    df_gold_estado
    .filter(col("ESTADO") != "BR")
    .orderBy(col("TOTAL_DESPESAS").desc())
)

In [0]:
df_gold_estado.createOrReplaceTempView("gold_estado")

### Tabela Gold Analítica (Flat)

Após a construção do esquema estrela, foi criada uma tabela analítica desnormalizada a partir da `FACT_DESPESA` e de suas dimensões. A estrutura reúne em uma única tabela as informações de candidato, estado, categoria de despesa, data, eleitorado e valor da despesa, facilitando o consumo dos dados nas consultas SQL.

In [0]:
df_gold_flat = (
    df_fact_despesa.alias("f")
    .join(
        df_dim_candidato.alias("c"),
        col("f.ID_CANDIDATO") == col("c.ID_CANDIDATO"),
        "left"
    )
    .join(
        df_dim_estado.alias("e"),
        col("f.ID_ESTADO") == col("e.ID_ESTADO"),
        "left"
    )
    .join(
        df_dim_categoria_despesa.alias("d"),
        col("f.ID_CATEGORIA_DESPESA") == col("d.ID_CATEGORIA_DESPESA"),
        "left"
    )
    .join(
        df_dim_data.alias("dt"),
        col("f.ID_DATA") == col("dt.ID_DATA"),
        "left"
    )
    .select(
        col("f.ID_DESPESA"),
        col("f.ID_CANDIDATO"),
        col("c.CANDIDATO"),
        col("c.PARTIDO"),
        col("c.CARGO"),
        col("e.ID_ESTADO"),
        col("e.ESTADO"),
        col("e.TOTAL_ELEITORES"),
        col("d.ID_CATEGORIA_DESPESA"),
        col("d.CATEGORIA_DESPESA"),
        col("d.DESPESA_DETALHADA"),
        col("dt.ID_DATA"),
        col("dt.DATA_DESPESA"),
        col("dt.ANO"),
        col("dt.MES"),
        col("dt.DIA"),
        col("dt.TRIMESTRE"),
        col("f.VALOR_DESPESA")
    )
)

display(df_gold_flat.limit(10))

In [0]:
df_gold_flat.createOrReplaceTempView("gold_despesa_analitica")

### Persistência da camada Gold
As estruturas da camada Gold foram persistidas no ambiente Databricks como tabelas, permitindo sua disponibilização no catálogo da plataforma e seu posterior consumo nas análises.

In [0]:
# Persistência das dimensões, tabela fato e tabela analítica
df_dim_candidato.write.mode("overwrite").saveAsTable("default.dim_candidato")
df_dim_estado.write.mode("overwrite").saveAsTable("default.dim_estado")
df_dim_categoria_despesa.write.mode("overwrite").saveAsTable("default.dim_categoria_despesa")
df_dim_data.write.mode("overwrite").saveAsTable("default.dim_data")
df_fact_despesa.write.mode("overwrite").saveAsTable("default.fact_despesa")
df_gold_flat.write.mode("overwrite").saveAsTable("default.gold_despesa_analitica")

### Validação da persistência Gold
As tabelas da camada Gold podem ser consultadas no catálogo do Databricks após a persistência.

In [0]:
%sql
SHOW TABLES IN default;

### Solução dos problemas propostos

A partir do processamento e da modelagem das bases de candidaturas, despesas eleitorais e eleitorado, foi construída uma estrutura analítica em esquema estrela, permitindo consolidar as informações em diferentes dimensões e facilitar sua consulta.

Com os dados preparados na camada Gold, torna-se possível realizar análises em SQL sobre a distribuição das despesas eleitorais, os candidatos envolvidos, as categorias de gastos e a relação entre despesas e eleitorado. A tabela analítica `gold_despesa_analitica` foi criada para reunir essas informações em uma única estrutura e passou a ser utilizada diretamente nas perguntas de negócio, facilitando o consumo e a comparação dos resultados.

### 1. Quais são os 5 estados com maior valor total de despesas eleitorais contratadas em 2026?

In [0]:
%sql
SELECT
    ESTADO,
    SUM(VALOR_DESPESA) AS TOTAL_DESPESAS
FROM gold_despesa_analitica
WHERE ESTADO <> 'BR'
GROUP BY ESTADO
ORDER BY TOTAL_DESPESAS DESC
LIMIT 5;

### Análise

Os resultados mostram que **São Paulo apresenta o maior valor total de despesas eleitorais contratadas, com R$ 505,97 milhões**, seguido por **Rio de Janeiro (R$ 278,53 milhões)**, **Minas Gerais (R$ 266,35 milhões)**, **Bahia (R$ 215,21 milhões)** e **Paraná (R$ 182,01 milhões)**.

Considerando apenas os cinco estados apresentados, o total de despesas é de aproximadamente **R$ 1,45 bilhão**. São Paulo concentra cerca de **34,9% desse valor** e registra um volume aproximadamente **1,8 vez maior que o Rio de Janeiro**, segundo colocado. Os resultados evidenciam uma concentração relevante das despesas entre as maiores unidades federativas.

### 1.1 Quais são os 3 candidatos com maior valor total de despesas contratadas em cada um desses 5 estados?


In [0]:
%sql
WITH top5_estados AS (
    SELECT
        ESTADO
    FROM gold_despesa_analitica
    WHERE ESTADO <> 'BR'
    GROUP BY ESTADO
    ORDER BY SUM(VALOR_DESPESA) DESC
    LIMIT 5
),

despesas_por_candidato AS (
    SELECT
        g.ESTADO,
        g.ID_CANDIDATO,
        g.CANDIDATO,
        g.PARTIDO,
        g.CARGO,
        SUM(g.VALOR_DESPESA) AS TOTAL_DESPESAS_CANDIDATO
    FROM gold_despesa_analitica g
    INNER JOIN top5_estados t
        ON g.ESTADO = t.ESTADO
    GROUP BY
        g.ESTADO,
        g.ID_CANDIDATO,
        g.CANDIDATO,
        g.PARTIDO,
        g.CARGO
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY ESTADO
            ORDER BY TOTAL_DESPESAS_CANDIDATO DESC
        ) AS RANK
    FROM despesas_por_candidato
)

SELECT
    ESTADO,
    RANK,
    CANDIDATO,
    PARTIDO,
    CARGO,
    TOTAL_DESPESAS_CANDIDATO
FROM ranking
WHERE RANK <= 3
ORDER BY ESTADO, RANK;

### Análise

Os resultados mostram diferenças na distribuição das despesas entre os três candidatos com maiores valores em cada estado. Em **São Paulo**, o primeiro colocado concentra aproximadamente **59,3%** do valor total dos três candidatos analisados, apresentando uma diferença significativa em relação ao segundo e ao terceiro colocados. Em **Bahia e Rio de Janeiro**, o primeiro candidato também representa cerca de metade das despesas dos três primeiros.

Já em **Paraná**, os valores dos dois primeiros candidatos são bastante próximos, enquanto em **Minas Gerais** observa-se uma distribuição mais equilibrada entre os três primeiros colocados. A análise permite, assim, observar como as despesas contratadas se distribuem entre os principais candidatos de cada estado, além de evidenciar diferentes níveis de concentração entre essas candidaturas.

### **2.** Quais são as 3 principais origens das despesas eleitorais contratadas em 2026?


In [0]:
%sql
SELECT
    CATEGORIA_DESPESA,
    SUM(VALOR_DESPESA) AS TOTAL_DESPESAS
FROM gold_despesa_analitica
GROUP BY CATEGORIA_DESPESA
ORDER BY TOTAL_DESPESAS DESC
LIMIT 3;

### Análise

Entre as três principais origens de despesas, **Publicidade por materiais impressos** apresenta o maior valor, com aproximadamente **R$ 652,76 milhões**, seguida por **Serviços prestados por terceiros**, com **R$ 600,40 milhões**. Os valores das duas categorias são próximos, com uma diferença de aproximadamente **R$ 52,36 milhões**.

Já **Atividades de militância e mobilização de rua** totalizam aproximadamente **R$ 46,69 milhões**, valor consideravelmente inferior aos dois primeiros. Somadas, as duas principais categorias representam cerca de **96,4% do total das três maiores origens**, indicando forte concentração das despesas nessas duas categorias de propaganda direta.

### 2.1 Quais são as 3 principais descrições de despesas dentro de cada uma das 3 principais origens?

In [0]:
%sql
WITH top3_origens AS (
    SELECT
        CATEGORIA_DESPESA
    FROM gold_despesa_analitica
    GROUP BY CATEGORIA_DESPESA
    ORDER BY SUM(VALOR_DESPESA) DESC
    LIMIT 3
),

despesas_por_origem AS (
    SELECT
        g.CATEGORIA_DESPESA,
        g.DESPESA_DETALHADA,
        SUM(g.VALOR_DESPESA) AS TOTAL_DESPESAS
    FROM gold_despesa_analitica g
    INNER JOIN top3_origens t
        ON g.CATEGORIA_DESPESA = t.CATEGORIA_DESPESA
    GROUP BY
        g.CATEGORIA_DESPESA,
        g.DESPESA_DETALHADA
),

ranking AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY CATEGORIA_DESPESA
            ORDER BY TOTAL_DESPESAS DESC
        ) AS RANK
    FROM despesas_por_origem
)

SELECT
    CATEGORIA_DESPESA,
    RANK,
    DESPESA_DETALHADA,
    TOTAL_DESPESAS
FROM ranking
WHERE RANK <= 3
ORDER BY CATEGORIA_DESPESA, RANK;

### Análise

Dentro das três principais origens de despesas, observa-se que cada categoria apresenta uma concentração em determinados tipos de gasto. Em **Atividades de militância e mobilização de rua**, a maior despesa é **Cabo Eleitoral**, com aproximadamente **R$ 19,54 milhões**, seguida por **Panfletagem (R$ 7,73 milhões)** e **Militância (R$ 7,35 milhões)**.

Em **Publicidade por materiais impressos**, destacam-se **Santinhos**, com aproximadamente **R$ 44,69 milhões**, seguidos por **Wind Banner (R$ 2,95 milhões)** e **Windbanner (R$ 1,95 milhão)**. Já em **Serviços prestados por terceiros**, os maiores valores estão concentrados em **Serviços especializados de comunicação e marketing (R$ 31,0 milhões)**, **Serviços de planejamento e marketing político (R$ 13,1 milhões)** e **Serviços de Marketing Digital (R$ 10,0 milhões)**. Os resultados mostram quais tipos específicos de despesas possuem maior participação dentro de cada uma das principais categorias.

### 3. Qual é o custo das despesas eleitorais contratadas, por eleitor, em cada estado brasileiro em 2026?


In [0]:
%sql
WITH estado AS (
    SELECT
        ESTADO,
        SUM(VALOR_DESPESA) AS TOTAL_DESPESAS,
        MAX(TOTAL_ELEITORES) AS TOTAL_ELEITORES
    FROM gold_despesa_analitica
    WHERE ESTADO <> 'BR'
    GROUP BY ESTADO
)

SELECT
    ESTADO,
    TOTAL_DESPESAS,
    TOTAL_ELEITORES,
    ROUND(
        TOTAL_DESPESAS / NULLIF(TOTAL_ELEITORES, 0),
        2
    ) AS CUSTO_POR_ELEITOR
FROM estado
ORDER BY CUSTO_POR_ELEITOR DESC;

### Análise

Os resultados mostram diferenças relevantes no custo das despesas eleitorais contratadas por eleitor entre os estados. **Roraima (RR)** apresenta o maior valor, com **R$ 172,49 por eleitor**, seguido por **Amapá (R$ 110,31)** e **Acre (R$ 69,68)**. Na sequência aparecem **Rondônia (R$ 65,74)** e **Tocantins (R$ 59,77)**.

Essa diferença mostra que, em alguns estados, há uma **maior intensidade de recursos contratados em relação ao tamanho do eleitorado**. Roraima, por exemplo, apresenta um custo por eleitor aproximadamente **56% superior ao Amapá**. Esse indicador pode servir como ponto de partida para investigar por que determinadas unidades federativas concentram mais recursos por eleitor, embora, isoladamente, não permita concluir que maiores gastos resultem em maior alcance ou impacto sobre os eleitores.

### Considerações Finais

### Análise consolidada dos problemas levantados

A análise conjunta das perguntas de negócio evidencia que o principal aspecto observado não está apenas no volume das despesas eleitorais, mas na forma como esses recursos se distribuem entre estados, candidatos e categorias de gasto. Os resultados demonstram diferenças relevantes no volume total contratado e, principalmente, no custo das despesas em relação ao tamanho do eleitorado de cada estado.

Esse cenário revela uma forte assimetria na aplicação dos recursos, com concentração das despesas em determinadas unidades federativas, candidatos e categorias. Embora os dados não permitam avaliar a efetividade ou o impacto desses gastos sobre os eleitores, as diferenças observadas indicam um ponto relevante para investigação: compreender quais fatores explicam essas disparidades e como o volume de recursos contratados se relaciona com a escala de cada eleitorado.

Esse levantamento também abre espaço para questionamentos relacionados ao acompanhamento, à transparência e à distribuição dos recursos empregados no processo eleitoral, especialmente diante das diferenças observadas entre as unidades federativas e entre as categorias de despesas.

De forma geral, o trabalho permitiu responder aos questionamentos definidos no início do projeto, consolidando as informações em uma estrutura analítica que facilita a consulta e a interpretação dos dados.

### Pontos de Melhoria

Uma das principais oportunidades de evolução do projeto seria o tratamento mais aprofundado dos registros incompletos ou anômalos identificados nas bases de origem. Embora esses registros tenham sido preservados na camada Silver para manter a fidelidade aos dados disponibilizados pelo TSE, uma etapa adicional de tratamento poderia classificar, corrigir ou excluir registros inadequados antes da consolidação da camada Gold, aumentando a precisão das métricas e análises realizadas.

Outra melhoria seria a automatização da atualização das bases. Devido ao volume dos arquivos e às limitações do ambiente Databricks utilizado no projeto, a coleta e o processamento foram realizados manualmente diretamente no Databricks, com os códigos disponibilizados no GitHub. Em uma implementação produtiva, seria desejável automatizar a ingestão e o processamento periódico dos dados, garantindo maior atualização e reprodutibilidade do pipeline, principalmente em períodos eleitorais, nos quais informações atualizadas podem contribuir para análises e acompanhamento do processo eleitoral.

Por fim, poderia ser implementado um processo de monitoramento da qualidade dos dados a cada nova carga. Indicadores como quantidade de nulos, registros anômalos, chaves sem correspondência, duplicidades e variações no volume das bases poderiam ser verificados automaticamente, permitindo identificar alterações ou problemas na fonte antes que afetem as análises.

### Autoavaliação

Em um âmbito pessoal, este foi um projeto extremamente enriquecedor e prazeroso de realizar. O módulo foi uma introdução a um nível de programação com o qual eu ainda não estava acostumado, mas fiquei muito feliz por ter tido a oportunidade de praticar, aprender e desenvolver análises que, no início do módulo, eu não imaginava ser capaz de construir.

Independentemente do resultado final, estou muito feliz com a minha entrega e, principalmente, com o meu desenvolvimento ao longo do projeto. 

Consegui escolher um tema às vésperas de um momento importante, trabalhar com dados reais e construir uma análise que pode ser útil para pessoas interessadas no assunto. Provavelmente em alguns meses, vou enxergar diversas coisas que poderiam ser melhoradas, mas acredito que isso faz parte do processo de adaptação a uma nova área. 

Saio deste módulo mais confiante nos meus conhecimentos e ainda mais entusiasmado para continuar aprendendo e me desenvolvendo com um próximo desafio.